# 🩺 Continuous 3D Internal Bioheat Visualizer (PINN + PyVista)

Welcome to your **interactive 3D medical visualizer**! 

This notebook loads your successfully trained **PINN state weights** and your **volumetric finite element meshes (`.msh`)** to project the continuous, deep internal temperature field inside the breast. You can rotate, zoom, and **slice the breast in half** to inspect the deep tumor hyperthermia core!

### ⚠️ Camera Perspective Note (Frontal vs. Sagittal Views)
* **Sagittal (Side) View (`camera_position = 'xy'`):** The patient was scanned standing upright. In the IMF-registered space, the chest wall is vertical, and the breast projects horizontally. The side view shows this standing profile.
* **Frontal (Forward) View (`camera_position = 'zy'`):** To view the breast upright and facing directly at you (like a standard frontal thermogram), look down the depth axis by setting the camera position to `'zy'` or `'yz'`.

---

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pyvista as pv
from pathlib import Path

# Set Jupyter rendering backend for interactive/static notebook widgets
pv.set_jupyter_backend('static') 

# Choose your Patient ID here!
PATIENT_ID = "Patient_1"
RESULTS_DIR = Path("results")
PATIENT_DIR = RESULTS_DIR / PATIENT_ID

print(f"📂 Loading directory: {PATIENT_DIR.absolute()}")

## 🧠 1. Load the PINN Model
We reconstruct the **`BioheatPINN`** architecture and load the PyTorch weights (`_pinn.pth`) solved by your inverse solver.

In [ ]:
class BioheatPINN(nn.Module):
    def __init__(self, hidden=256, depth=6):
        super().__init__()
        layers = [nn.Linear(3, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)

        # Learnable tumour parameters
        self.x_t   = nn.Parameter(torch.tensor([0.0]))
        self.y_t   = nn.Parameter(torch.tensor([0.0]))
        self.z_t   = nn.Parameter(torch.tensor([0.0]))
        self.r_t   = nn.Parameter(torch.tensor([10.0]))    # mm
        self.Q_max = nn.Parameter(torch.tensor([5000.0]))  # W/m³

    def forward(self, xyz):
        return self.net(xyz).squeeze(-1)

# 1. Re-initialize model architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BioheatPINN().to(device)

# 2. Load PyTorch weights
weights_path = PATIENT_DIR / f"{PATIENT_ID}_pinn.pth"
if not weights_path.exists():
    raise FileNotFoundError(f"❌ Weights not found for {PATIENT_ID} at {weights_path}")

pinn_state = torch.load(weights_path, map_location=device)
model.load_state_dict(pinn_state)
model.eval()
print("✅ PINN neural network weights successfully loaded!")

## 🕸️ 2. Read the Volumetric Mesh
We load the **Gmsh solid tetrahedron model (`.msh`)** of the breast.

In [ ]:
mesh_path = PATIENT_DIR / f"{PATIENT_ID}.msh"
if not mesh_path.exists():
    raise FileNotFoundError(f"❌ tetrahedral mesh not found at {mesh_path}")

# Load volumetric mesh with PyVista
grid = pv.read(mesh_path)
print(f"✅ Volumetric Tet Mesh successfully loaded!")
print(grid)

## 🌡️ 3. Evaluate PINN Continuous Temperatures on Mesh Vertices
Unlike classic discrete solvers, the **PINN is a continuous spatial mapping**. We evaluate the network at every single node vertex inside the 3D volume mesh to solve for the continuous internal temperature distribution.

In [ ]:
# Extract vertex coordinates from the mesh (in mm)
vertices = np.array(grid.points)

# Calculate bounding box directly from the volumetric mesh vertices!
bbox_min = vertices.min(axis=0)
bbox_max = vertices.max(axis=0)
print(f"📍 Normalization Bounding Box: Min={bbox_min}, Max={bbox_max}")

# Normalize coordinates to [-1, 1] matching PINN training limits
centre = (bbox_max + bbox_min) / 2
extent = (bbox_max - bbox_min) / 2 + 1e-8
verts_norm = (vertices - centre) / extent

# Convert to torch tensor and evaluate
with torch.no_grad():
    xyz_tensor = torch.tensor(verts_norm, dtype=torch.float32, device=device)
    T_pred = model(xyz_tensor).cpu().numpy().flatten()

# Append solved temperatures as a scalar field to the PyVista mesh grid
grid["Temperature (°C)"] = T_pred
print(f"🔥 Continuous volumetric temperature field successfully mapped to {len(T_pred)} vertices!")
print(f"Temperature Range: Min={T_pred.min():.2f}°C, Max={T_pred.max():.2f}°C")

## ✂️ 4. Interactive Volumetric Slice Rendering (Thesis Gold Visual!)
To validate the internal heat field and inspect the deep tumor core, we **slice the 3D breast mesh** along the sagittal or coronal plane.

In [ ]:
# Set up standard plot
plotter = pv.Plotter(window_size=[1000, 800])
plotter.background_color = "#0f0f0f" # Sleek clinical dark mode

# Create an interactive slicing plane through the center of the breast volume
    "# normal=[1, 0, 0] slices along the sagittal plane (vertical split, separating left/right)\n",
sliced_grid = grid.slice(normal=[1, 0, 0], origin=grid.center)

# Add the solid outer silhouette outline with transparency
plotter.add_mesh(
    grid.outline(),
    color="#ffffff",
    line_width=1.5
)

# Add the slice showing internal continuous heat gradients
plotter.add_mesh(
    sliced_grid,
    scalars="Temperature (°C)",
    cmap="hot",  # Excellent bio-thermal color representation
    clim=[28.0, 37.0], # Absolute biological temperature range
    show_scalar_bar=True,
    scalar_bar_args={"title": "Internal Temp (°C)", "shadow": True}
)

# Add studio lighting for premium visual quality
plotter.add_light(pv.Light(position=(200, 200, 500), intensity=0.8))

# Set camera view. Use 'zy' for frontal view, or 'xy' for side (sagittal) view!
plotter.camera_position = 'xy'
plotter.camera.zoom(1.2)

# Render the plot
plotter.show()

## 🗺️ 5. 3D Surface Convective Overlay
Inspect the continuous temperature profile mapped directly on the **outer skin surface boundary**.

In [ ]:
plotter = pv.Plotter(window_size=[1000, 800])
plotter.background_color = "#0d0e15" # Dark indigo premium space

# Add the outer surface skin mesh colored by temperature
plotter.add_mesh(
    grid,
    scalars="Temperature (°C)",
    cmap="hot",
    clim=[28.0, 37.0],
    opacity=0.95,
    show_scalar_bar=True
)

# Use 'zy' for frontal, or 'xy' for side!
plotter.camera_position = 'zy'
plotter.camera.zoom(1.3)
plotter.show()